# PyTorch for LLMs 04: The Modern Training Lifecycle

> **What you finished last time:** explicit batch policy, reload parity, and resumable state.
> **The gap this chapter closes:** a training loop alone does not protect validation evidence, model modes, stopping decisions, or recovery state.
> **What this notebook delivers:** a compact trainer assembled from readable functions, plus best-model and resumable checkpoints.
> **What this chapter is not:** a trainer-framework tutorial; Lightning and \`torch.compile\` remain optional translations.

**Guiding question:** Which lifecycle decisions must remain visible even when a framework later automates them?

| Complaint | Refinement | Measured evidence |
|---|---|---|
| preprocessing can leak | split before normalization | different statistics |
| validation can accidentally learn | separate no-gradient loop | no validation grads |
| dropout changes behavior | call \`train()\` and \`eval()\` | repeated outputs differ/agree |
| last epoch may not be best | early stop on validation | best epoch recorded |
| one checkpoint cannot serve two jobs | save best and resumable forms | both reload |

\`\`\`mermaid
flowchart LR
    A["Split raw data"] --> B["Fit train statistics"]
    B --> C["Train batches"]
    C --> D["Validation"]
    D --> E{"Improved?"}
    E -->|yes| F["Best-model checkpoint"]
    E -->|no| G["Patience counter"]
    G --> H{"Stop?"}
    H -->|no| C
    H -->|yes| I["Reload best"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
\`\`\`


In [ ]:
# -- Imports, reproducibility, and configuration ------------------------------
from dataclasses import asdict, dataclass
from pathlib import Path
import json
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(53)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@dataclass
class TrainConfig:
    seed: int = 53
    batch_size: int = 24
    learning_rate: float = 0.02
    max_epochs: int = 80
    patience: int = 10

config = TrainConfig()
print(f"Configuration: {asdict(config)}")


## 1. Split before calculating normalization statistics

**Predict:** If validation has a shifted temperature distribution, will all-data statistics match train-only statistics? No. Measuring the difference exposes the leak.


In [ ]:
# -- Create a nonlinear fuel-efficiency dataset -------------------------------
generator = torch.Generator().manual_seed(config.seed)
n_samples = 180
engine_load = torch.rand(n_samples, generator=generator)
temperature = 1.8 * torch.rand(n_samples, generator=generator) - 0.8
X_raw = torch.stack([engine_load, temperature], dim=1)
y_raw = (
    1.2
    + 1.7 * engine_load
    - 0.8 * temperature
    + 0.55 * engine_load * temperature
    + 0.10 * torch.randn(n_samples, generator=generator)
).unsqueeze(1)
print(f"Raw contracts: X={tuple(X_raw.shape)}, y={tuple(y_raw.shape)}")


In [ ]:
# -- Split first, then fit preprocessing on training rows only ----------------
indices = torch.randperm(n_samples, generator=generator)
train_idx, val_idx, test_idx = indices[:108], indices[108:144], indices[144:]
X_train_raw, X_val_raw, X_test_raw = X_raw[train_idx], X_raw[val_idx], X_raw[test_idx]
y_train, y_val, y_test = y_raw[train_idx], y_raw[val_idx], y_raw[test_idx]

train_mean = X_train_raw.mean(dim=0)
train_std = X_train_raw.std(dim=0)
all_mean = X_raw.mean(dim=0)
mean_gap = (train_mean - all_mean).abs().max().item()
print(f"Max train-only versus all-data mean gap: {mean_gap:.4f}")
print("Only train statistics will transform validation and test rows.")


In [ ]:
# -- Normalize features and targets without crossing evidence boundaries -------
y_mean = y_train.mean(dim=0)
y_std = y_train.std(dim=0)
normalize_X = lambda values: (values - train_mean) / train_std
normalize_y = lambda values: (values - y_mean) / y_std
X_train, X_val, X_test = map(normalize_X, (X_train_raw, X_val_raw, X_test_raw))
y_train_n, y_val_n, y_test_n = map(normalize_y, (y_train, y_val, y_test))
print(f"Train feature mean after normalization: {X_train.mean(dim=0).tolist()}")
print(f"Validation feature mean remains evidence: {X_val.mean(dim=0).tolist()}")


## 2. DataLoaders state the batch policy


In [ ]:
# -- Build loaders with explicit shuffle policies -----------------------------
train_dataset = TensorDataset(X_train, y_train_n)
val_dataset = TensorDataset(X_val, y_val_n)
test_dataset = TensorDataset(X_test, y_test_n)
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    generator=torch.Generator().manual_seed(config.seed),
)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)
sample_X, sample_y = next(iter(train_loader))
print(f"Batch contract: X={tuple(sample_X.shape)}, y={tuple(sample_y.shape)}")


## 3. Model mode is part of the result

Dropout is stochastic during training and disabled during evaluation.

**Predict:** Will two passes over the same batch match in train mode? Usually not. In evaluation mode they should.


In [ ]:
# -- Define a compact model with mode-sensitive dropout -----------------------
class FuelEfficiencyRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(16, 1),
        )

    def forward(self, inputs):
        return self.network(inputs)

model = FuelEfficiencyRegressor().to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters())}")


In [ ]:
# -- Measure train/eval behavior on identical input ---------------------------
mode_probe = sample_X[:6].to(device)
model.train()
train_a, train_b = model(mode_probe), model(mode_probe)
model.eval()
with torch.inference_mode():
    eval_a, eval_b = model(mode_probe), model(mode_probe)
print(f"Train-mode max repeat gap: {(train_a - train_b).abs().max().item():.6f}")
print(f"Eval-mode max repeat gap: {(eval_a - eval_b).abs().max().item():.6f}")
assert torch.equal(eval_a, eval_b)


## 4. Decompose the trainer into readable lifecycle pieces

Mixed precision is useful on supported accelerators, but it is not a CPU requirement. The scaler is enabled only on CUDA.


In [ ]:
# -- Configure loss, optimizer, scheduler, and mixed-precision boundary -------
loss_function = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
print(f"Mixed precision enabled: {use_amp}")
print(f"Initial learning rate: {optimizer.param_groups[0]['lr']}")


In [ ]:
# -- Define one training epoch without orchestration noise --------------------
def train_one_epoch(model, dataloader, optimizer, scaler):
    model.train()
    total_loss = 0.0
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, enabled=use_amp):
            loss = loss_function(model(batch_X), batch_y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(batch_X)
    return total_loss / len(dataloader.dataset)

print("Defined train_one_epoch: gradients and updates occur only here.")


In [ ]:
# -- Define validation as measurement with no updates -------------------------
def evaluate(model, dataloader):
    model.eval()
    total_loss = 0.0
    with torch.inference_mode():
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            loss = loss_function(model(batch_X), batch_y)
            total_loss += loss.item() * len(batch_X)
    return total_loss / len(dataloader.dataset)

print("Defined evaluate: eval mode plus inference_mode prevents learning.")


### Code walkthrough: the two loops

- Training selects stochastic layer behavior, clears gradients, backpropagates, and updates.
- Validation selects deterministic behavior and constructs no gradient graph.
- Both weight batch losses by batch size before averaging, so a smaller final batch is not over-weighted.


In [ ]:
# -- Define one small early-stopping state machine ----------------------------
class EarlyStopping:
    def __init__(self, patience):
        self.patience = patience
        self.best = float("inf")
        self.bad_epochs = 0

    def update(self, value):
        improved = value < self.best
        if improved:
            self.best = value
            self.bad_epochs = 0
        else:
            self.bad_epochs += 1
        return improved, self.bad_epochs >= self.patience

early_stopping = EarlyStopping(config.patience)
print(f"Early stopping patience: {early_stopping.patience} validation checks")


## 5. Best-model and resumable checkpoints serve different jobs

\`\`\`mermaid
flowchart LR
    A["Validation improves"] --> B["Best model\nweights + preprocessing"]
    C["Epoch completes"] --> D["Resume state\nmodel + optimizer + scheduler + epoch"]
    B --> E["Inference"]
    D --> F["Continue training"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
\`\`\`


In [ ]:
# -- Prepare checkpoint paths and history -------------------------------------
artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
best_path = artifact_dir / "best_model.pt"
resume_path = artifact_dir / "resume_training.pt"
history = {"train": [], "validation": [], "learning_rate": []}
best_epoch = -1
print(f"Checkpoint directory: {artifact_dir.resolve()}")


In [ ]:
# -- Assemble the visible training lifecycle ----------------------------------
for epoch in range(config.max_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
    validation_loss = evaluate(model, val_loader)
    scheduler.step(validation_loss)
    improved, should_stop = early_stopping.update(validation_loss)
    history["train"].append(train_loss)
    history["validation"].append(validation_loss)
    history["learning_rate"].append(optimizer.param_groups[0]["lr"])
    if improved:
        best_epoch = epoch
        torch.save({
            "model_state": model.state_dict(),
            "train_mean": train_mean,
            "train_std": train_std,
            "target_mean": y_mean,
            "target_std": y_std,
            "config": asdict(config),
        }, best_path)
    if should_stop:
        break

print(f"Ran {len(history['train'])} epochs; best validation epoch={best_epoch + 1}")
print(f"Validation loss: {history['validation'][0]:.4f} -> {min(history['validation']):.4f}")


In [ ]:
# -- Save resumable state separately from the best inference state ------------
torch.save({
    "epoch": epoch,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "scheduler_state": scheduler.state_dict(),
    "early_stopping": vars(early_stopping),
    "config": asdict(config),
}, resume_path)
print(f"Best-model checkpoint: {best_path}")
print(f"Resumable checkpoint: {resume_path}")


In [ ]:
# -- Plot only recorded history ------------------------------------------------
epochs = range(1, len(history["train"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, history["train"], label="train")
axes[0].plot(epochs, history["validation"], label="validation")
axes[0].set(xlabel="epoch", ylabel="normalized MSE", title="Recorded losses")
axes[0].legend()
axes[1].step(epochs, history["learning_rate"], where="post")
axes[1].set(xlabel="epoch", ylabel="learning rate", title="Scheduler decisions")
for axis in axes:
    axis.grid(alpha=0.3)
plt.show()
print("The plots contain actual lifecycle measurements, not illustrative curves.")


## 6. Reload the best checkpoint and infer in original units


In [ ]:
# -- Reload best state into a fresh model and verify test inference -----------
best_checkpoint = torch.load(best_path, map_location=device, weights_only=True)
best_model = FuelEfficiencyRegressor().to(device)
best_model.load_state_dict(best_checkpoint["model_state"])
best_model.eval()
with torch.inference_mode():
    test_prediction_n = best_model(X_test.to(device)).cpu()
test_prediction = test_prediction_n * best_checkpoint["target_std"] + best_checkpoint["target_mean"]
test_mae = (test_prediction - y_test).abs().mean().item()
print(f"Test MAE in original fuel units: {test_mae:.4f}")
print(f"Prediction shape: {tuple(test_prediction.shape)}; target shape: {tuple(y_test.shape)}")


In [ ]:
# -- Verify the resumable checkpoint has every continuation component ---------
resume_checkpoint = torch.load(resume_path, map_location=device, weights_only=True)
required = {"epoch", "model_state", "optimizer_state", "scheduler_state", "early_stopping", "config"}
missing = required - set(resume_checkpoint)
print(f"Missing resumable fields: {sorted(missing)}")
assert not missing


## Optional translations: Lightning and \`torch.compile\`

Lightning maps \`train_one_epoch\`, \`evaluate\`, optimizer configuration, and checkpoint callbacks into framework hooks. \`torch.compile\` may optimize execution after correctness is established. Neither removes the need to understand split integrity, mode changes, loss shapes, or checkpoint contents.

### Your turn: change only patience


In [ ]:
# -- CHANGE THIS: inspect the stopping rule without retraining -----------------
YOUR_PATIENCE = 4
simulated = EarlyStopping(YOUR_PATIENCE)
sequence = [1.0, 0.8, 0.81, 0.82, 0.83, 0.84]
stop_at = None
for index, value in enumerate(sequence, start=1):
    _, stopped = simulated.update(value)
    if stopped:
        stop_at = index
        break
print(f"With patience={YOUR_PATIENCE}, the simulated run stops at check {stop_at}.")
assert stop_at is not None


## Scorecard and coverage ledger

| Tier | Covered here |
|---|---|
| Built and measured | split-before-normalization, loaders, train/eval modes, separate loops, scheduler, AMP boundary, early stopping, best/resume checkpoints, reload, original-unit inference |
| Explained or illustrated | Lightning mapping and compile-after-correctness boundary |
| Named and out of scope | distributed training, gradient clipping, accumulation, experiment trackers, distributed checkpointing |

**Common failure modes:** fitting normalization on all rows, validating in train mode, updating on validation, early-stopping on training loss, and confusing "best for inference" with "enough to resume."

**Durable summary:** automation may hide lifecycle code, but it cannot remove lifecycle responsibilities.

**Next boundary:** the route is ready for sequence memory and tokenization. Those chapters will reuse the same axis, mode, and checkpoint contracts at language-model shapes.
